## 🌐 **Google Drive Connection**

In [1]:
from google.colab import drive
drive.mount("/gdrive", force_remount=True)
current_dir = "/gdrive/My\\ Drive/Project/Challenge\\ 2-v2"
%cd $current_dir

Mounted at /gdrive
/gdrive/My Drive/Project/Challenge 2-v2


## ⚙️ **Libraries Import**

In [2]:
# Set seed for reproducibility
SEED = 42

# Import necessary libraries
import os

# Set environment variables before importing modules
os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['MPLCONFIGDIR'] = os.getcwd() + '/configs/'

# Suppress warnings
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=Warning)

# Import necessary modules
import logging
import random
import numpy as np

# Set seeds for random number generators in NumPy and Python
np.random.seed(SEED)
random.seed(SEED)

# Import PyTorch
import torch
torch.manual_seed(SEED)
from torch import nn
from torchsummary import summary
from torch.utils.tensorboard import SummaryWriter
import torchvision
from torchvision import datasets, transforms
from torch.utils.data import TensorDataset, DataLoader, Dataset

# Configurazione di TensorBoard e directory
logs_dir = "tensorboard"
!pkill -f tensorboard
%load_ext tensorboard
!mkdir -p models

if torch.cuda.is_available():
    device = torch.device("cuda")
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True
else:
    device = torch.device("cpu")

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {device}")

# Import other libraries
import copy
import shutil
from itertools import product
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.model_selection import train_test_split
from PIL import Image
import matplotlib.gridspec as gridspec
import cv2
from tqdm import tqdm

# Configure plot display settings
sns.set(font_scale=1.4)
sns.set_style('white')
plt.rc('font', size=14)
%matplotlib inline

#Lion optimizer
%pip install lion-pytorch

PyTorch version: 2.9.0+cu126
Device: cpu


## 🦠 **Data Cleaning**

In [3]:
# Directory for dataset storage
DATA_DIR = "./train_data"
CSV_PATH = "./train_labels.csv"
CLEAN_DATA_DIR = "./clean_train_data"
CLEAN_CSV_DIR = "./clean_train_labels.csv"
PATCH_DIR = "./patch"
PATCH_CSV = "./patch.csv"

In [4]:
def create_clean_dataset(original_df, source_dir, dest_dir, dest_csv, threshold=0.005):
    """
    Filters the dataset and copies valid images/masks to a new folder.
    """
    print(f"🧹 Creating clean dataset in: {dest_dir}")

    # Create the destination directory if it doesn't exist
    os.makedirs(dest_dir, exist_ok=True)

    clean_rows = []
    rejected_count = 0

    print(f"🕵️ Scanning {len(original_df)} images...")

    for idx in tqdm(range(len(original_df))):
        try:
            # Get filenames
            img_name = original_df.iloc[idx, 0]
            mask_name = img_name.replace("img", "mask")

            src_img_path = os.path.join(source_dir, img_name)
            src_mask_path = os.path.join(source_dir, mask_name)

            # 1. READ IMAGE
            img = cv2.imread(src_img_path)
            if img is None: continue

            # 2. FILTER LOGIC (Shrek/Slime Detector)
            b, g, r = img[:,:,0], img[:,:,1], img[:,:,2]

            # Define foreground (not white background)
            bg_mask = (b > 215) & (g > 215) & (r > 215)
            fg_count = np.sum(~bg_mask)

            is_clean = True

            # If image isn't empty, check for poison
            if fg_count > 0:
                bad_pixels = ((g > b + 20) | (g > r + 20)) & (~bg_mask)
                if (np.sum(bad_pixels) / fg_count) > threshold:
                    is_clean = False
                    rejected_count += 1

            # 3. SAVE IF CLEAN
            if is_clean:
                # Copy Image
                shutil.copy(src_img_path, os.path.join(dest_dir, img_name))

                # Copy Mask (Crucial! We need this for the Dataset class later)
                if os.path.exists(src_mask_path):
                    shutil.copy(src_mask_path, os.path.join(dest_dir, mask_name))

                # Keep the label record
                clean_rows.append(original_df.iloc[idx])

        except Exception as e:
            print(f"Error processing {img_name}: {e}")

    # Save the new CSV
    df_clean = pd.DataFrame(clean_rows)
    df_clean.to_csv(dest_csv, index=False)

    print(f"\n✅ Done!")
    print(f"❌ Rejected {rejected_count} poisoned images.")
    print(f"📂 Saved {len(df_clean)} clean images/masks to '{dest_dir}'")

    return df_clean

In [4]:
if os.path.exists(CLEAN_CSV_DIR) and os.path.exists(CLEAN_DATA_DIR):
    print(f"✅ Found cached clean dataset at: {CLEAN_DATA_DIR}")
    print("   Skipping scan and using existing files.")

    # Load the clean list
    df_clean = pd.read_csv(CLEAN_CSV_DIR)

else:
    print("⚠️ Clean dataset not found. Creating it now...")
    original_df = pd.read_csv(CSV_PATH)

    # Run creation (Filter + Copy)
    df_clean = create_clean_dataset(original_df, DATA_DIR, CLEAN_DATA_DIR, CLEAN_CSV_DIR)


✅ Found cached clean dataset at: ./clean_train_data
   Skipping scan and using existing files.


## 🔎 **Data Loading**

In [5]:
# 2. Define the dictionary to translate them
label_mapping = {
    "Luminal A": 0,
    "Luminal B": 1,
    "HER2(+)": 2,
    "Triple negative": 3
}

# 3. Apply the translation
df_clean['label'] = df_clean['label'].map(label_mapping)

# 4. Check for any mistakes (NaN means a spelling mismatch)
if df_clean['label'].isnull().any():
    print("WARNING: Some labels didn't match! Check spelling.")
    print(df_clean[df_clean['label'].isnull()])
else:
    print("Success! Labels converted to numbers:", df_clean['label'].unique())

Success! Labels converted to numbers: [3 1 0 2]


# __Patch extraction__

In [31]:
import cv2
import numpy as np
import os
import pandas as pd
from tqdm import tqdm

def extract_patches_for_classification(df, source_dir, dest_dir, patch_size=256, stride=128, tissue_threshold=0.1):
    """
    Fixed extraction logic:
    1. Uses count_nonzero (works for masks with values 1 or 255).
    2. Uses smaller patches (256x256) to fit thin biopsy strips better.
    """
    img_dest = os.path.join(dest_dir, "images")
    mask_dest = os.path.join(dest_dir, "masks")
    os.makedirs(img_dest, exist_ok=True)
    os.makedirs(mask_dest, exist_ok=True)

    patch_data = []

    print(f"🔪 Slicing {len(df)} images...")
    print(f"⚙️ Params: Patch={patch_size}, Stride={stride}, Min Ratio={tissue_threshold}")

    for idx in tqdm(range(len(df))):
        try:
            row = df.iloc[idx]
            img_name = row[0]

            # Construct mask name
            mask_name = os.path.splitext(img_name)[0].replace("img", "mask") + ".png"
            label = row.get('label', row[1] if len(row)>1 else 'Unknown')

            img_path = os.path.join(source_dir, img_name)
            mask_path = os.path.join(source_dir, mask_name)

            # Fallback path check
            if not os.path.exists(mask_path):
                 mask_path = os.path.join(source_dir, img_name.replace("img", "mask"))

            img = cv2.imread(img_path)
            mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

            if img is None or mask is None: continue

            # Pad image
            h, w, _ = img.shape
            pad_h = (patch_size - h % patch_size) % patch_size
            pad_w = (patch_size - w % patch_size) % patch_size

            if pad_h > 0 or pad_w > 0:
                img = cv2.copyMakeBorder(img, 0, pad_h, 0, pad_w, cv2.BORDER_CONSTANT, value=[255,255,255])
                mask = cv2.copyMakeBorder(mask, 0, pad_h, 0, pad_w, cv2.BORDER_CONSTANT, value=0)

            h_new, w_new, _ = img.shape

            patch_id = 0
            for y in range(0, h_new - patch_size + 1, stride):
                for x in range(0, w_new - patch_size + 1, stride):

                    img_patch = img[y:y+patch_size, x:x+patch_size]
                    mask_patch = mask[y:y+patch_size, x:x+patch_size]

                    # --- FIXED RATIO CALCULATION ---
                    # Simply count non-black pixels.
                    # This works for value 1 (invisible mask) AND value 255 (white mask).
                    tissue_pixels = np.count_nonzero(mask_patch)
                    total_pixels = patch_size * patch_size
                    tissue_ratio = tissue_pixels / total_pixels

                    if tissue_ratio >= tissue_threshold:
                        p_img_name = f"{os.path.splitext(img_name)[0]}_p{patch_id}.png"
                        p_mask_name = f"{os.path.splitext(mask_name)[0]}_p{patch_id}.png"

                        cv2.imwrite(os.path.join(img_dest, p_img_name), img_patch)
                        cv2.imwrite(os.path.join(mask_dest, p_mask_name), mask_patch)

                        patch_data.append({
                            "filename": p_img_name,
                            "maskname": p_mask_name,
                            "label": label,
                            "original_image": img_name
                        })

                    patch_id += 1

        except Exception as e:
            print(f"Error on {img_name}: {e}")

    new_df = pd.DataFrame(patch_data)
    new_df.to_csv(PATCH_CSV, index=False)

    print(f"\n✅ Done!")
    print(f"📊 Generated {len(new_df)} patches.")

    return new_df

In [8]:
df_clean = pd.read_csv(CLEAN_CSV_DIR)

# We feed the CLEAN dataframe into the patcher
df_patches = extract_patches_for_classification(
    df_clean,
    CLEAN_DATA_DIR,
    PATCH_DIR,
    patch_size=224,      # Native ResNet size (no resizing needed)
    stride=56,           # High Overlap (75%) -> Generates 4x more patches
)

🔪 Slicing 577 images...
⚙️ Params: Patch=224, Stride=56, Min Ratio=0.1


100%|██████████| 577/577 [05:20<00:00,  1.80it/s]


✅ Done!
📊 Generated 13448 patches.


# __Train-Validation splitting__
Since we are using patches, we must guarantee that all the patches from one single image goes entirely in the train set or in the validation set. Otherwise, we have a big data likage problem

In [7]:
import pandas as pd
from sklearn.model_selection import train_test_split

# 1. Load your patch dataset
df_patches = pd.read_csv(PATCH_CSV)

# 2. Group by Original Slide to avoid Data Leakage
# We get one row per patient/slide
slide_data = df_patches.groupby('original_image')['label'].first().reset_index()

# 3. Split the SLIDES (not the patches)
train_slides, val_slides = train_test_split(
    slide_data,
    test_size=0.15,
    stratify=slide_data['label'], # Ensures we keep class balance (e.g. equal % of Triple Negative)
    random_state=SEED
)

# 4. Filter the main dataframe based on the split
train_df = df_patches[df_patches['original_image'].isin(train_slides['original_image'])].reset_index(drop=True)
val_df = df_patches[df_patches['original_image'].isin(val_slides['original_image'])].reset_index(drop=True)

print(f"✅ Data Split Complete (Patient-Level Safety)")
print(f"   Train Slides: {len(train_slides)} | Train Patches: {len(train_df)}")
print(f"   Val Slides:   {len(val_slides)} | Val Patches:   {len(val_df)}")

✅ Data Split Complete (Patient-Level Safety)
   Train Slides: 375 | Train Patches: 11442
   Val Slides:   67 | Val Patches:   2006


# __Training__
Before starting the training loop, we need to preprocess our data and do some data aumentation.

In [8]:
# 1. Define your mapping (Must match your Class Weights logic!)
label_mapper = {
    "Luminal A": 0,
    "Luminal B": 1,
    "HER2(+)": 2,       # Check if your CSV has "HER2+" or "HER2(+)"
    "Triple negative": 3
}

# 2. Check what is currently in your dataframe to be safe
print("Labels before fixing:", train_df['label'].unique())

# 3. Apply the mapping
# Using .map() converts the strings to integers
train_df['label'] = train_df['label'].map(label_mapper)
val_df['label'] = val_df['label'].map(label_mapper)

# 4. Handle any NaNs (Mismatched spellings)
if train_df['label'].isnull().any():
    print("❌ ERROR: Some labels did not match the dictionary!")
    print(train_df[train_df['label'].isnull()]['label'].unique())
    # Quick fix: drop them or map them manually
    train_df = train_df.dropna(subset=['label'])
    val_df = val_df.dropna(subset=['label'])

# 5. Convert to Integer explicitly
train_df['label'] = train_df['label'].astype(int)
val_df['label'] = val_df['label'].astype(int)

print("✅ Labels fixed:", train_df['label'].unique())

Labels before fixing: ['Luminal B' 'Luminal A' 'HER2(+)' 'Triple negative']
✅ Labels fixed: [1 0 2 3]


In [58]:
import torch
from torch.utils.data import Dataset, DataLoader
import cv2
import numpy as np
import os
import albumentations as A
from albumentations.pytorch import ToTensorV2
from tqdm import tqdm

class HistologyDataset(Dataset):
    def __init__(self, df, root_dir, transform=None):
        self.df = df
        self.root_dir = root_dir
        self.transform = transform

        # --- CACHING: Load everything into RAM immediately ---
        print(f"⏳ Pre-loading {len(df)} images into RAM...")
        self.images = []
        self.labels = []

        for idx in tqdm(range(len(df))):
            row = df.iloc[idx]

            img_path = os.path.join(self.root_dir, "images", row['filename'])
            mask_path = os.path.join(self.root_dir, "masks", row['maskname'])

            # Load Image
            image = cv2.imread(img_path)
            if image is None:
                continue
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

            # Load Mask
            mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
            if mask is None:
                continue

            # --- PRE-COMPUTE BLACKOUT STRATEGY ---
            # We do this ONCE here, instead of doing it every epoch.
            # This saves massive amounts of CPU time during training.
            _, binary_mask = cv2.threshold(mask, 127, 1, cv2.THRESH_BINARY)
            mask_3d = np.repeat(binary_mask[:, :, np.newaxis], 3, axis=2)
            image = image * mask_3d

            # Store the pre-processed image
            self.images.append(image)
            self.labels.append(row['label'])

        print("✅ Cache complete! Training will now be instant.")

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        # 1. Fetch from RAM (Zero latency)
        image = self.images[idx]
        label = self.labels[idx]

        # 2. Apply Augmentations (This is the only work left for the CPU)
        if self.transform:
            augmented = self.transform(image=image)
            image = augmented['image']

        return image, torch.tensor(label, dtype=torch.long)

# --- DEFINE TRANSFORMS ---
# Training: Heavy Augmentation to prevent overfitting
train_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.CoarseDropout(max_holes=8, max_height=32, max_width=32, min_holes=4, fill_value=0, p=0.5),
    A.ElasticTransform(alpha=1, sigma=50, alpha_affine=50, p=0.2), # Deformation
    A.RandomBrightnessContrast(p=0.2),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

# Validation: Normalize only
val_transform = A.Compose([
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

# --- CREATE LOADERS ---
train_ds = HistologyDataset(train_df, PATCH_DIR, transform=train_transform)
val_ds = HistologyDataset(val_df, PATCH_DIR, transform=val_transform)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=4)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=4)

⏳ Pre-loading 11442 images into RAM...


100%|██████████| 11442/11442 [02:26<00:00, 77.93it/s]


✅ Cache complete! Training will now be instant.
⏳ Pre-loading 2006 images into RAM...


100%|██████████| 2006/2006 [00:23<00:00, 87.20it/s]

✅ Cache complete! Training will now be instant.


## 🧮 **Network Parameters**

In [46]:
from sklearn.utils.class_weight import compute_class_weight
import torch
import numpy as np

# 1. Get labels directly from the DataFrame (Instant)
# We use the dataframe 'train_df' that you created during the train_test_split
all_train_labels = train_df['label'].values

print("Calculating class weights from DataFrame...")

# 2. Compute weights
class_weights_np = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(all_train_labels),
    y=all_train_labels
)

# 3. Move to GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
class_weights = torch.tensor(class_weights_np, dtype=torch.float).to(device)

print(f"Class Weights applied: {class_weights}")

Calculating class weights from DataFrame...
Class Weights applied: tensor([0.9434, 0.6166, 1.1516, 2.2226], device='cuda:0')


In [47]:
# Set up TensorBoard logging and save model architecture
experiment_name = "patches_resnet"
writer = SummaryWriter("./"+logs_dir+"/"+experiment_name)
#x = torch.randn(1, input_shape[0], input_shape[1], input_shape[2]).to(device)
#writer.add_graph(cnn_model, x)

# Define optimizer with L2 regularization
#optimizer = torch.optim.AdamW(cnn_model.parameters(), lr=LEARNING_RATE, weight_decay=L2_LAMBDA)

# Enable mixed precision training for GPU acceleration
#scaler = torch.amp.GradScaler(enabled=(device.type == 'cuda'))

# **Using ResNet18**

In [67]:
import torchvision.models as models
import torch.nn as nn
import torch.optim as optim
from torch.cuda.amp import GradScaler

# 1. Load Model
cnn_model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

# 2. ULTIMATE FREEZE (Linear Probing Mode)
# Freeze EVERY layer in the ResNet backbone.
for param in cnn_model.parameters():
    param.requires_grad = False

# 3. The Classification Head
# Since the body is frozen, we need a slightly robust head to interpret the features.
num_features = cnn_model.fc.in_features
cnn_model.fc = nn.Sequential(
    nn.Linear(num_features, 256), # Intermediate layer to mix features
    nn.ReLU(),
    nn.Dropout(0.5),              # Dropout to prevent head overfitting
    nn.Linear(256, 4)             # Final Output
)

cnn_model = cnn_model.to(device)

# 4. Optimizer: AdamW
# IMPORTANT: We only pass the Head parameters. The body is ignored.
optimizer = optim.AdamW(cnn_model.fc.parameters(), lr=1e-3, weight_decay=0.01)

# 5. Loss Function (Keep Label Smoothing)
# It helps prevent the model from becoming too confident.
criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)

# 6. Scheduler & Scaler
scaler = GradScaler()
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.1, patience=3
)

print("✅ Model Configured: Linear Probing (Backbone Frozen, Head Training only)")

✅ Model Configured: Linear Probing (Backbone Frozen, Head Training only)


## 🧠 **Model Training**

In [68]:
# Initialize best model tracking variables
best_model = None
best_performance = float('-inf')

In [69]:
def train_one_epoch(model, train_loader, criterion, optimizer, scaler, device, l1_lambda=0, l2_lambda=0):
    """
    Perform one complete training epoch through the entire training dataset.

    Args:
        model (nn.Module): The neural network model to train
        train_loader (DataLoader): PyTorch DataLoader containing training data batches
        criterion (nn.Module): Loss function (e.g., CrossEntropyLoss, MSELoss)
        optimizer (torch.optim): Optimization algorithm (e.g., Adam, SGD)
        scaler (GradScaler): PyTorch's gradient scaler for mixed precision training
        device (torch.device): Computing device ('cuda' for GPU, 'cpu' for CPU)
        l1_lambda (float): Lambda for L1 regularization
        l2_lambda (float): Lambda for L2 regularization

    Returns:
        tuple: (average_loss, f1 score) - Training loss and f1 score for this epoch
    """
    model.train()  # Set model to training mode

    running_loss = 0.0
    all_predictions = []
    all_targets = []

    # Iterate through training batches
    for batch_idx, (inputs, targets) in enumerate(train_loader):
        # Move data to device (GPU/CPU)
        inputs, targets = inputs.to(device), targets.to(device)

        # Clear gradients from previous step
        optimizer.zero_grad(set_to_none=True)

        # Forward pass with mixed precision (if CUDA available)
        with torch.amp.autocast(device_type=device.type, enabled=(device.type == 'cuda')):
            logits = model(inputs)
            loss = criterion(logits, targets)

            # Add L1 and L2 regularization
            # l1_norm = sum(p.abs().sum() for p in model.parameters())
            # l2_norm = sum(p.pow(2).sum() for p in model.parameters())
            # loss = loss + l1_lambda * l1_norm + l2_lambda * l2_norm


        # Backward pass with gradient scaling
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        # Accumulate metrics
        running_loss += loss.item() * inputs.size(0)
        predictions = logits.argmax(dim=1)
        all_predictions.append(predictions.cpu().numpy())
        all_targets.append(targets.cpu().numpy())

    # Calculate epoch metrics
    epoch_loss = running_loss / len(train_loader.dataset)
    epoch_f1 = f1_score(
        np.concatenate(all_targets),
        np.concatenate(all_predictions),
        average='weighted'
    )

    return epoch_loss, epoch_f1

In [70]:
def validate_one_epoch(model, val_loader, criterion, device):
    """
    Perform one complete validation epoch through the entire validation dataset.

    Args:
        model (nn.Module): The neural network model to evaluate (must be in eval mode)
        val_loader (DataLoader): PyTorch DataLoader containing validation data batches
        criterion (nn.Module): Loss function used to calculate validation loss
        device (torch.device): Computing device ('cuda' for GPU, 'cpu' for CPU)

    Returns:
        tuple: (average_loss, accuracy) - Validation loss and accuracy for this epoch

    Note:
        This function automatically sets the model to evaluation mode and disables
        gradient computation for efficiency during validation.
    """
    model.eval()  # Set model to evaluation mode

    running_loss = 0.0
    all_predictions = []
    all_targets = []

    # Disable gradient computation for validation
    with torch.no_grad():
        for inputs, targets in val_loader:
            # Move data to device
            inputs, targets = inputs.to(device), targets.to(device)

            # Forward pass with mixed precision (if CUDA available)
            with torch.amp.autocast(device_type=device.type, enabled=(device.type == 'cuda')):
                logits = model(inputs)
                loss = criterion(logits, targets)

            # Accumulate metrics
            running_loss += loss.item() * inputs.size(0)
            predictions = logits.argmax(dim=1)
            all_predictions.append(predictions.cpu().numpy())
            all_targets.append(targets.cpu().numpy())

    # Calculate epoch metrics
    epoch_loss = running_loss / len(val_loader.dataset)
    epoch_accuracy = f1_score(
        np.concatenate(all_targets),
        np.concatenate(all_predictions),
        average='weighted'
    )

    return epoch_loss, epoch_accuracy

In [71]:
def log_metrics_to_tensorboard(writer, epoch, train_loss, train_f1, val_loss, val_f1, model):
    """
    Log training metrics and model parameters to TensorBoard for visualization.

    Args:
        writer (SummaryWriter): TensorBoard SummaryWriter object for logging
        epoch (int): Current epoch number (used as x-axis in TensorBoard plots)
        train_loss (float): Training loss for this epoch
        train_f1 (float): Training f1 score for this epoch
        val_loss (float): Validation loss for this epoch
        val_f1 (float): Validation f1 score for this epoch
        model (nn.Module): The neural network model (for logging weights/gradients)

    Note:
        This function logs scalar metrics (loss/f1 score) and histograms of model
        parameters and gradients, which helps monitor training progress and detect
        issues like vanishing/exploding gradients.
    """
    # Log scalar metrics
    writer.add_scalar('Loss/Training', train_loss, epoch)
    writer.add_scalar('Loss/Validation', val_loss, epoch)
    writer.add_scalar('F1/Training', train_f1, epoch)
    writer.add_scalar('F1/Validation', val_f1, epoch)

    # Log model parameters and gradients
    for name, param in model.named_parameters():
        if param.requires_grad:
            # Check if the tensor is not empty before adding a histogram
            if param.numel() > 0:
                writer.add_histogram(f'{name}/weights', param.data, epoch)
            if param.grad is not None:
                # Check if the gradient tensor is not empty before adding a histogram
                if param.grad.numel() > 0:
                    if param.grad is not None and torch.isfinite(param.grad).all():
                        writer.add_histogram(f'{name}/gradients', param.grad.data, epoch)

In [72]:
def fit(model, train_loader, val_loader, epochs, criterion, optimizer, scaler, device,
        l1_lambda=0, l2_lambda=0, patience=0, evaluation_metric="val_f1", mode='max',
        restore_best_weights=True, writer=None, verbose=1, experiment_name="", scheduler=None):
    """
    Train the neural network model on the training data and validate on the validation data.

    Args:
        model (nn.Module): The neural network model to train
        train_loader (DataLoader): PyTorch DataLoader containing training data batches
        val_loader (DataLoader): PyTorch DataLoader containing validation data batches
        epochs (int): Number of training epochs
        criterion (nn.Module): Loss function (e.g., CrossEntropyLoss, MSELoss)
        optimizer (torch.optim): Optimization algorithm (e.g., Adam, SGD)
        scaler (GradScaler): PyTorch's gradient scaler for mixed precision training
        device (torch.device): Computing device ('cuda' for GPU, 'cpu' for CPU)
        l1_lambda (float): L1 regularization coefficient (default: 0)
        l2_lambda (float): L2 regularization coefficient (default: 0)
        patience (int): Number of epochs to wait for improvement before early stopping (default: 0)
        evaluation_metric (str): Metric to monitor for early stopping (default: "val_f1")
        mode (str): 'max' for maximizing the metric, 'min' for minimizing (default: 'max')
        restore_best_weights (bool): Whether to restore model weights from best epoch (default: True)
        writer (SummaryWriter, optional): TensorBoard SummaryWriter object for logging (default: None)
        verbose (int, optional): Frequency of printing training progress (default: 10)
        experiment_name (str, optional): Experiment name for saving models (default: "")

    Returns:
        tuple: (model, training_history) - Trained model and metrics history
    """

    # Initialize metrics tracking
    training_history = {
        'train_loss': [], 'val_loss': [],
        'train_f1': [], 'val_f1': []
    }

    # Configure early stopping if patience is set
    if patience > 0:
        patience_counter = 0
        best_metric = float('-inf') if mode == 'max' else float('inf')
        best_epoch = 0

    print(f"Training {epochs} epochs...")

    # Main training loop: iterate through epochs
    for epoch in range(1, epochs + 1):

        # Forward pass through training data, compute gradients, update weights
        train_loss, train_f1 = train_one_epoch(
            model, train_loader, criterion, optimizer, scaler, device, l1_lambda, l2_lambda
        )

        # Evaluate model on validation data without updating weights
        val_loss, val_f1 = validate_one_epoch(
            model, val_loader, criterion, device
        )

        # Store metrics for plotting and analysis
        training_history['train_loss'].append(train_loss)
        training_history['val_loss'].append(val_loss)
        training_history['train_f1'].append(train_f1)
        training_history['val_f1'].append(val_f1)

        # Write metrics to TensorBoard for visualization
        if writer is not None:
            log_metrics_to_tensorboard(
                writer, epoch, train_loss, train_f1, val_loss, val_f1, model
            )

        # --- SCHEDULER STEP ---
        if scheduler is not None:
            # ReduceLROnPlateau needs the validation metric (loss)
            if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
                scheduler.step(val_loss)
            else:
                scheduler.step()

        # Print progress every N epochs or on first epoch
        if verbose > 0:
            if epoch % verbose == 0 or epoch == 1:
                print(f"Epoch {epoch:3d}/{epochs} | "
                    f"Train: Loss={train_loss:.4f}, F1 Score={train_f1:.4f} | "
                    f"Val: Loss={val_loss:.4f}, F1 Score={val_f1:.4f}")

        # Early stopping logic: monitor metric and save best model
        if patience > 0:
            current_metric = training_history[evaluation_metric][-1]
            is_improvement = (current_metric > best_metric) if mode == 'max' else (current_metric < best_metric)

            if is_improvement:
                best_metric = current_metric
                best_epoch = epoch
                torch.save(model.state_dict(), "models/"+experiment_name+'_model.pt')
                patience_counter = 0
            else:
                patience_counter += 1
                if patience_counter >= patience:
                    print(f"Early stopping triggered after {epoch} epochs.")
                    break

    # Restore best model weights if early stopping was used
    if restore_best_weights and patience > 0:
        model.load_state_dict(torch.load("models/"+experiment_name+'_model.pt'))
        print(f"Best model restored from epoch {best_epoch} with {evaluation_metric} {best_metric:.4f}")

    # Save final model if no early stopping
    if patience == 0:
        torch.save(model.state_dict(), "models/"+experiment_name+'_model.pt')

    # Close TensorBoard writer
    if writer is not None:
        writer.close()

    return model, training_history

In [73]:
# Ensure class_weights are defined (from previous steps)
criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)

# print(f"🚀 Unleashing the Lion on {device}...")

cnn_model, training_history = fit(
    model=cnn_model,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=30,                  # Lion is efficient, but give it time to settle
    criterion=criterion,
    optimizer=optimizer,        # Passing the Lion
    scaler=scaler,
    device=device,
    patience=8,                 # Slightly higher patience allows Lion to escape local minima
    evaluation_metric="val_f1", # Track F1 Score
    mode='max',                 # We want F1 to go UP
    experiment_name="resnet18_lion_opt",
    writer=writer,
    scheduler=scheduler
)

# Update best model if current performance is superior
if training_history['val_f1'][-1] > best_performance:
    best_model = cnn_model
    best_performance = training_history['val_f1'][-1]

🚀 Unleashing the Lion on cuda...
Training 30 epochs...
Epoch   1/30 | Train: Loss=1.4028, F1 Score=0.2841 | Val: Loss=1.4249, F1 Score=0.1840
Epoch   2/30 | Train: Loss=1.3656, F1 Score=0.3101 | Val: Loss=1.4231, F1 Score=0.2352
Epoch   3/30 | Train: Loss=1.3410, F1 Score=0.3323 | Val: Loss=1.4512, F1 Score=0.1698
Epoch   4/30 | Train: Loss=1.3234, F1 Score=0.3419 | Val: Loss=1.4699, F1 Score=0.2866
Epoch   5/30 | Train: Loss=1.2994, F1 Score=0.3607 | Val: Loss=1.4206, F1 Score=0.3288
Epoch   6/30 | Train: Loss=1.2916, F1 Score=0.3677 | Val: Loss=1.4426, F1 Score=0.2567
Epoch   7/30 | Train: Loss=1.2792, F1 Score=0.3766 | Val: Loss=1.4065, F1 Score=0.3130
Epoch   8/30 | Train: Loss=1.2726, F1 Score=0.3737 | Val: Loss=1.4547, F1 Score=0.2668
Epoch   9/30 | Train: Loss=1.2681, F1 Score=0.3824 | Val: Loss=1.4441, F1 Score=0.3133
Epoch  10/30 | Train: Loss=1.2673, F1 Score=0.3878 | Val: Loss=1.4743, F1 Score=0.2986
Epoch  11/30 | Train: Loss=1.2614, F1 Score=0.3928 | Val: Loss=1.4466, F1 S

KeyboardInterrupt: 

In [77]:
import torch
import torch.optim as optim
from torch.cuda.amp import GradScaler

# 1. LOAD THE BEST MODEL FROM STAGE 1
# Ensure you point to the file saved by your previous run
# It might be named "resnet18_finetuned_model.pt" or similar based on your experiment_name
stage1_path = "./models/resnet18_lion_opt_model.pt"

print(f"📥 Loading Stage 1 weights from {stage1_path}...")
state_dict = torch.load(stage1_path)
cnn_model.load_state_dict(state_dict)

# 2. UNFREEZE LAYER 4 (The "Unlock")
# Layers 1, 2, 3 remain frozen (Basic textures are fine).
# Layer 4 adapts to Cancer shapes.
for param in cnn_model.layer4.parameters():
    param.requires_grad = True

# 3. OPTIMIZER: DIFFERENTIAL LEARNING RATES
# This is the secret sauce.
# Body (Layer 4): Extremely low LR (1e-5) -> Nudge it gently.
# Head (FC): Low LR (1e-4) -> Refine the classification.
optimizer = optim.AdamW([
    {'params': cnn_model.layer4.parameters(), 'lr': 1e-5},
    {'params': cnn_model.fc.parameters(), 'lr': 1e-4}
], weight_decay=0.01)

# 4. RESET SCHEDULER
# We start fresh.
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.1, patience=3
)

print("🚀 Starting Stage 2: Fine-Tuning Layer 4...")

# 5. Fine tune on layer 4
trained_model, history = fit(
    model=cnn_model,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=20,
    criterion=criterion,
    optimizer=optimizer,
    scaler=scaler,
    device=device,
    patience=5,
    evaluation_metric="val_f1",
    mode='max',
    experiment_name="resnet18_stage2_finetune", # Save as a new file!
    writer=writer,
    scheduler=scheduler
)

📥 Loading Stage 1 weights from ./models/resnet18_lion_opt_model.pt...
🚀 Starting Stage 2: Fine-Tuning Layer 4...
Training 20 epochs...
Epoch   1/20 | Train: Loss=1.2193, F1 Score=0.4231 | Val: Loss=1.4790, F1 Score=0.2977
Epoch   2/20 | Train: Loss=1.1720, F1 Score=0.4594 | Val: Loss=1.5123, F1 Score=0.3160
Epoch   3/20 | Train: Loss=1.1431, F1 Score=0.4806 | Val: Loss=1.4883, F1 Score=0.3233
Epoch   4/20 | Train: Loss=1.1165, F1 Score=0.5036 | Val: Loss=1.5123, F1 Score=0.3446
Epoch   5/20 | Train: Loss=1.0918, F1 Score=0.5298 | Val: Loss=1.5272, F1 Score=0.3283


KeyboardInterrupt: 

In [ ]:
# @title Plot Hitory
# Create a figure with two side-by-side subplots (two columns)
fig, (ax1, ax2) = plt.subplots(nrows=1, ncols=2, figsize=(18, 5))

# Plot of training and validation loss on the first axis
ax1.plot(training_history['train_loss'], label='Training loss', alpha=0.3, color='#ff7f0e', linestyle='--')
ax1.plot(training_history['val_loss'], label='Validation loss', alpha=0.9, color='#ff7f0e')
ax1.set_title('Loss')
ax1.legend()
ax1.grid(alpha=0.3)

# Plot of training and validation accuracy on the second axis
ax2.plot(training_history['train_f1'], label='Training f1', alpha=0.3, color='#ff7f0e', linestyle='--')
ax2.plot(training_history['val_f1'], label='Validation f1', alpha=0.9, color='#ff7f0e')
ax2.set_title('F1 Score')
ax2.legend()
ax2.grid(alpha=0.3)

# Adjust the layout and display the plot
plt.tight_layout()
plt.subplots_adjust(right=0.85)
plt.show()

In [ ]:
# @title Plot Confusion Matrix
# Collect predictions and ground truth labels
val_preds, val_targets = [], []
with torch.no_grad():  # Disable gradient computation for inference
    for xb, yb in val_loader:
        xb = xb.to(device)

        # Forward pass: get model predictions
        logits = cnn_model(xb)
        preds = logits.argmax(dim=1).cpu().numpy()

        # Store batch results
        val_preds.append(preds)
        val_targets.append(yb.numpy())

# Combine all batches into single arrays
val_preds = np.concatenate(val_preds)
val_targets = np.concatenate(val_targets)

# Calculate overall validation metrics
val_acc = accuracy_score(val_targets, val_preds)
val_prec = precision_score(val_targets, val_preds, average='weighted')
val_rec = recall_score(val_targets, val_preds, average='weighted')
val_f1 = f1_score(val_targets, val_preds, average='weighted')
print(f"Accuracy over the validation set: {val_acc:.4f}")
print(f"Precision over the validation set: {val_prec:.4f}")
print(f"Recall over the validation set: {val_rec:.4f}")
print(f"F1 score over the validation set: {val_f1:.4f}")

# Generate confusion matrix for detailed error analysis
cm = confusion_matrix(val_targets, val_preds)

# Create numeric labels for heatmap annotation
labels = np.array([f"{num}" for num in cm.flatten()]).reshape(cm.shape)

# Visualise confusion matrix
plt.figure(figsize=(8, 7))
sns.heatmap(cm, annot=labels, fmt='',
            cmap='Blues')
plt.xlabel('Predicted labels')
plt.ylabel('True labels')
plt.title('Confusion Matrix — Validation Set')
plt.tight_layout()
plt.show()

In [ ]:
# @title Activation visualisation
def get_activation(name):
    """Creates a hook function to capture and store layer outputs."""
    def hook(model, input, output):
        activations[name] = output.detach()
    return hook


def find_last_conv_layer(model):
    """
    Identifies the final Conv2D layer in the model architecture.
    """
    last_conv_name = None
    for name, module in model.named_modules():
        if isinstance(module, nn.Conv2d):
            last_conv_name = name

    if last_conv_name is None:
        raise ValueError("No Conv2D layer found in the model.")
    return last_conv_name


def visualize(model, X, y, unique_labels, num_images=50, display_activations=True, display_all_conv_layers=False):
    """
    Visualises model predictions and internal activations for a random test image.
    Uses PyTorch hooks to extract intermediate layer outputs.

    Args:
        display_all_conv_layers: If True, shows all conv layers. If False, shows only last conv of each block.
    """

    # --- 1. Select Image and Prepare Tensor ---

    # Randomly select an image from the dataset
    image_idx = np.random.randint(0, num_images)
    img_np = X[image_idx]
    label_np = y[image_idx]

    # Convert NumPy array to PyTorch tensor with correct dimensions
    # Transform from (H, W, C) to (N, C, H, W) format
    img_tensor = torch.from_numpy(img_np)
    img_tensor = img_tensor.permute(2, 0, 1)
    img_tensor = img_tensor.unsqueeze(0).to(device)

    # --- 2. Register Hooks and Make Prediction ---

    # Clear previous activations
    activations.clear()

    # Attach forward hooks to convolutional layers
    hooks = []
    conv_names = []

    # Iterate through all blocks in the features Sequential
    for block_idx, block in enumerate(model.features):
        # Find all Conv2d layers in this block
        conv_layers_in_block = []
        for layer_idx, layer in enumerate(block.block):
            if isinstance(layer, nn.Conv2d):
                conv_layers_in_block.append((layer_idx, layer))

        # Register hooks based on display_all_conv_layers flag
        if display_all_conv_layers:
            # Register hook for every Conv2d layer
            for layer_idx, conv_layer in conv_layers_in_block:
                hook_name = f'block{block_idx}_conv{layer_idx}'
                conv_names.append(hook_name)
                hooks.append(conv_layer.register_forward_hook(get_activation(hook_name)))
        else:
            # Register hook only for the last Conv2d layer in this block
            if conv_layers_in_block:
                layer_idx, conv_layer = conv_layers_in_block[-1]
                hook_name = f'block{block_idx}_conv{layer_idx}'
                conv_names.append(hook_name)
                hooks.append(conv_layer.register_forward_hook(get_activation(hook_name)))

    # Generate prediction with gradient tracking disabled
    model.eval()
    with torch.no_grad():
        logits = model(img_tensor)
        probabilities = torch.softmax(logits, dim=1)

    # Remove hooks after forward pass
    for hook in hooks:
        hook.remove()

    # Extract predicted class and confidence
    predictions = probabilities.cpu().numpy()
    class_int = np.argmax(predictions[0])
    class_str = unique_labels[class_int]

    # Extract true class (handle both one-hot encoded and integer labels)
    if label_np.ndim > 0 and len(label_np) > 1:
        # One-hot encoded
        true_class_int = np.argmax(label_np)
    else:
        # Already an integer index
        true_class_int = int(label_np)
    true_class_str = unique_labels[true_class_int]

    # --- 3. Plot Image and Prediction Bar ---

    # Create figure with custom layout
    fig = plt.figure(constrained_layout=True, figsize=(16, 4))
    gs = gridspec.GridSpec(1, 2, figure=fig, width_ratios=[1.5, 1.5], wspace=0)

    # Display original image with true label
    ax1 = fig.add_subplot(gs[0])
    ax1.set_title(f"True class: {true_class_str}", loc='left')
    if img_np.shape[-1] == 1:
        ax1.imshow(np.squeeze(img_np), cmap='bone', vmin=0., vmax=1.)
    else:
        ax1.imshow(np.squeeze(img_np), vmin=0., vmax=1.)
    ax1.axis('off')

    # Display class probability distribution
    ax2 = fig.add_subplot(gs[1])
    ax2.barh(unique_labels, np.squeeze(predictions, axis=0), color=plt.get_cmap('tab10').colors)
    ax2.set_title(f"Predicted class: {class_str} (Confidence: {max(np.squeeze(predictions[0])):.2f})", loc='left')
    ax2.grid(alpha=0.3)
    ax2.set_xlim(0.0, 1.0)
    plt.show()

    # --- 4. Plot Activations ---

    if display_activations:
        # Visualise activations for each registered layer
        for conv_name in conv_names:
            # Retrieve stored activations from hooks
            layer_activations = activations[conv_name]

            # Get number of channels
            num_channels = layer_activations.shape[1]

            # Display up to 16 feature maps per layer
            num_display = min(16, num_channels)

            # Calculate grid layout
            if num_display <= 8:
                rows, cols = 1, num_display
                figsize = (18, 3)
            else:
                rows, cols = 2, 8
                figsize = (18, 5)

            # Create subplot grid
            fig, axes = plt.subplots(rows, cols, figsize=figsize)

            # Flatten axes array for easier indexing
            if num_display > 1:
                axes = axes.flatten() if rows > 1 or cols > 1 else [axes]
            else:
                axes = [axes]

            # Plot each activation map
            for i in range(num_display):
                ax = axes[i]
                activation_map = layer_activations[0, i].cpu().numpy()
                ax.imshow(activation_map, cmap='bone', vmin=np.min(activation_map), vmax=np.max(activation_map))
                ax.axis('off')
                if i == 0:
                    ax.set_title(f'{conv_name} activations', loc='left')

            # Hide unused subplots
            for i in range(num_display, len(axes)):
                axes[i].axis('off')

            plt.tight_layout()
            plt.show()

# __Test data and submission generation__

In [38]:
import os
import cv2
import pandas as pd
from tqdm import tqdm

# --- CONFIG ---
TEST_DIR = './test_data/'      # Where your big test images are
TEMP_PATCH_DIR = './test_patches_temp/' # Where we save the small patches
PATCH_SIZE = 224
STRIDE = 56                    # Use the same dense stride (56) or loose (224)
                               # Stride 56 = More votes per slide (Recommended)
TISSUE_THRESH = 0.1            # Same 10% threshold as training

os.makedirs(os.path.join(TEMP_PATCH_DIR, "images"), exist_ok=True)
os.makedirs(os.path.join(TEMP_PATCH_DIR, "masks"), exist_ok=True)

test_files = [f for f in os.listdir(TEST_DIR) if f.startswith('img') and f.endswith(('.png', '.jpg'))]
patch_data = []

print(f"🔪 Slicing {len(test_files)} Test Slides...")

for filename in tqdm(test_files):
    # 1. Match Image and Mask
    img_path = os.path.join(TEST_DIR, filename)

    # Try generic mask replacement
    mask_name = filename.replace('img', 'mask').replace('.jpg', '.png')
    mask_path = os.path.join(TEST_DIR, mask_name)

    # Fallback if extension is different
    if not os.path.exists(mask_path):
        mask_name = filename.replace('img', 'mask') # Try keep original ext
        mask_path = os.path.join(TEST_DIR, mask_name)

    if not os.path.exists(mask_path):
        print(f"⚠️ Mask missing for {filename}, skipping...")
        continue

    # 2. Load
    image = cv2.imread(img_path)
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

    if image is None or mask is None: continue

    # 3. Slice
    h, w, _ = image.shape

    for y in range(0, h - PATCH_SIZE + 1, STRIDE):
        for x in range(0, w - PATCH_SIZE + 1, STRIDE):

            # Extract
            img_patch = image[y:y+PATCH_SIZE, x:x+PATCH_SIZE]
            mask_patch = mask[y:y+PATCH_SIZE, x:x+PATCH_SIZE]

            # Filter (Exact same logic as Training)
            tissue_area = (mask_patch > 0).sum()
            if (tissue_area / (PATCH_SIZE**2)) >= TISSUE_THRESH:

                # Save to disk
                patch_name = f"{filename[:-4]}_{y}_{x}.png"

                # We save valid patches exactly like we did for train_data
                cv2.imwrite(os.path.join(TEMP_PATCH_DIR, "images", patch_name), img_patch)
                cv2.imwrite(os.path.join(TEMP_PATCH_DIR, "masks", patch_name), mask_patch)

                # Record metadata
                patch_data.append({
                    'original_image': filename,
                    'filename': patch_name,
                    'maskname': patch_name # We saved them with same name
                })

# Create DataFrame
test_df = pd.DataFrame(patch_data)
print(f"✅ Generated {len(test_df)} test patches.")

🔪 Slicing 477 Test Slides...


100%|██████████| 477/477 [07:21<00:00,  1.08it/s]

✅ Generated 11607 test patches.


In [39]:
# --- CONFIGURATION ---
TEST_DIR = './test_data/'
MODEL_PATH = './models/resnet18_stage2_finetune_model.pt'
PATCH_SIZE = 224
STRIDE = 56   # Dense stride (same as training/validation) for maximum votes
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [40]:
import torch
import torch.nn as nn
import cv2
import numpy as np
import pandas as pd
import os
import albumentations as A
from albumentations.pytorch import ToTensorV2
from tqdm import tqdm
from scipy import stats

# 1. LABEL MAPPING
# Ensure these match your training class indices exactly
idx_to_label = {
    0: 'Luminal A',
    1: 'Luminal B',
    2: 'HER2(+)',
    3: 'Triple negative'
}

# 2. TRANSFORM (Normalize Only)
test_transform = A.Compose([
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

# 3. HELPER: ROBUST EXTRACTION
def extract_patches_with_fallback(img_path, mask_path, patch_size=224, stride=56):
    img = cv2.imread(img_path)

    # Handling missing/corrupt files
    if img is None:
        return None

    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

    # If mask is missing, create a fake full-white mask (assume all tissue)
    if mask is None:
        mask = np.ones((img.shape[0], img.shape[1]), dtype=np.uint8) * 255

    patches = []

    # Pad image to match patch size
    h, w, _ = img.shape
    pad_h = (patch_size - h % patch_size) % patch_size
    pad_w = (patch_size - w % patch_size) % patch_size
    if pad_h > 0 or pad_w > 0:
        img = cv2.copyMakeBorder(img, 0, pad_h, 0, pad_w, cv2.BORDER_CONSTANT, value=[255,255,255])
        mask = cv2.copyMakeBorder(mask, 0, pad_h, 0, pad_w, cv2.BORDER_CONSTANT, value=0)

    h_new, w_new, _ = img.shape

    # TRACKERS for Fallback
    best_fallback_patch = None
    max_tissue = -1

    # A. STANDARD LOOP (Strict Validation Logic)
    for y in range(0, h_new - patch_size + 1, stride):
        for x in range(0, w_new - patch_size + 1, stride):

            img_patch = img[y:y+patch_size, x:x+patch_size]
            mask_patch = mask[y:y+patch_size, x:x+patch_size]

            # Count Tissue
            tissue_pixels = np.count_nonzero(mask_patch)
            tissue_ratio = tissue_pixels / (patch_size * patch_size)

            # PREPARE BLACKOUT (Exact Training Logic)
            _, binary_mask = cv2.threshold(mask_patch, 127, 1, cv2.THRESH_BINARY)
            mask_3d = np.repeat(binary_mask[:, :, np.newaxis], 3, axis=2)
            final_patch = img_patch * mask_3d

            # Tensorize
            tensor_patch = test_transform(image=final_patch)['image']

            # STRICT FILTER: >10% Tissue (Like Validation)
            if tissue_ratio >= 0.10:
                patches.append(tensor_patch)

            # TRACK BEST (For Fallback)
            if tissue_pixels > max_tissue:
                max_tissue = tissue_pixels
                best_fallback_patch = tensor_patch

    # B. DECISION TIME
    if len(patches) > 0:
        # Scenario 1: Standard case, we found valid patches
        return torch.stack(patches)

    elif best_fallback_patch is not None and max_tissue > 0:
        # Scenario 2: Mask exists but no patch met 10% threshold.
        # We return the single "best" patch we found.
        return torch.stack([best_fallback_patch])

    else:
        # Scenario 3: Mask was 100% black/empty.
        # We FORCE a center crop on the image (ignoring the mask).
        cy, cx = h_new // 2, w_new // 2

        # Safe bounds calculation
        y1 = max(0, cy - patch_size // 2)
        x1 = max(0, cx - patch_size // 2)
        # Ensure we don't go out of bounds (unlikely due to padding, but safe)
        if y1 + patch_size > h_new: y1 = h_new - patch_size
        if x1 + patch_size > w_new: x1 = w_new - patch_size

        center_patch = img[y1:y1+patch_size, x1:x1+patch_size]

        # Transform and return
        tensor_patch = test_transform(image=center_patch)['image']
        return torch.stack([tensor_patch])

# 4. LOAD MODEL
print(f"🚀 Loading model from {MODEL_PATH}")

# 1. Load Base ResNet
model = torch.hub.load('pytorch/vision:v0.10.0', 'resnet18', weights=None)

# 2. DEFINE THE EXACT SAME HEAD AS TRAINING (Stage 2)
# We used: Linear(512->256) -> ReLU -> Dropout -> Linear(256->4)
num_features = model.fc.in_features
model.fc = nn.Sequential(
    nn.Linear(num_features, 256),  # fc.0 (Matches "Unexpected key fc.0.weight")
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(256, 4)              # fc.3 (Matches "Unexpected key fc.3.weight")
)

# 3. LOAD WEIGHTS (Strict Mode MUST pass now)
checkpoint = torch.load(MODEL_PATH, map_location=DEVICE)

# Fix 'module.' prefix if present
if list(checkpoint.keys())[0].startswith('module.'):
    checkpoint = {k.replace('module.', ''): v for k, v in checkpoint.items()}

try:
    model.load_state_dict(checkpoint, strict=True)
    print("✅ Weights loaded successfully! (Strict Mode Passed)")
except RuntimeError as e:
    print("❌ STILL FAILING:", e)

model.to(DEVICE)
model.eval()

Using cache found in /root/.cache/torch/hub/pytorch_vision_v0.10.0


🚀 Loading model from ./models/resnet18_stage2_finetune_model.pt
✅ Weights loaded successfully! (Strict Mode Passed)


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [41]:
from torch.utils.data import Dataset, DataLoader

# --- 5. INFERENCE LOOP (Using Saved Patches) ---

# A. Define Dataset for Saved Patches
# --- IMPROVED DATASET CLASS (HANDLES MISSING FILES) ---
class SavedPatchDataset(Dataset):
    def __init__(self, df, root_dir, transform=None):
        self.df = df
        self.root_dir = root_dir
        self.transform = transform
        # Create a black placeholder image once to use if a file is missing
        self.dummy_image = np.zeros((224, 224, 3), dtype=np.uint8)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # Construct paths
        img_path = os.path.join(self.root_dir, "images", row['filename'])
        mask_path = os.path.join(self.root_dir, "masks", row['maskname'])

        # 1. SAFETY CHECK: Does file exist?
        if not os.path.exists(img_path):
            # Print warning only once to avoid spamming (optional logic)
            # print(f"⚠️ Missing file: {row['filename']} - Returning black placeholder.")
            image = self.dummy_image
            mask = np.zeros((224, 224), dtype=np.uint8)
        else:
            # 2. LOAD
            image = cv2.imread(img_path)
            if image is None: # Double check if load failed
                image = self.dummy_image
                mask = np.zeros((224, 224), dtype=np.uint8)
            else:
                image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
                mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
                if mask is None: mask = np.zeros((224, 224), dtype=np.uint8)

        # 3. BLACKOUT
        _, binary_mask = cv2.threshold(mask, 127, 1, cv2.THRESH_BINARY)
        mask_3d = np.repeat(binary_mask[:, :, np.newaxis], 3, axis=2)
        image = image * mask_3d

        # 4. TRANSFORM
        if self.transform:
            augmented = self.transform(image=image)
            image = augmented['image']

        return image, row['original_image']

# B. Create Loader
# 'test_df' is the dataframe you created in the previous step (Block 1)
test_ds = SavedPatchDataset(test_df, TEMP_PATCH_DIR, transform=test_transform)
test_loader = DataLoader(test_ds, batch_size=128, shuffle=False, num_workers=2)

print(f"🕵️ Predicting on {len(test_df)} saved patches...")

# C. Main Prediction Loop
patch_predictions = []
model.eval()

with torch.no_grad():
    for images, slide_names in tqdm(test_loader):
        images = images.to(DEVICE)

        # Mixed Precision Prediction
        with torch.amp.autocast(device_type=DEVICE.type, enabled=(DEVICE.type == 'cuda')):
            logits = model(images)
            preds = torch.argmax(logits, dim=1).cpu().numpy()

        # Store individual patch result
        for slide, pred in zip(slide_names, preds):
            patch_predictions.append({'sample_index': slide, 'pred': pred})

# D. Aggregation (Majority Vote)
print("🗳️ Voting...")
raw_df = pd.DataFrame(patch_predictions)
results = []
predicted_slides = set()

if not raw_df.empty:
    for slide_name, group in raw_df.groupby('sample_index'):
        # Calculate Mode
        final_pred = stats.mode(group['pred'])[0]
        if isinstance(final_pred, np.ndarray): final_pred = final_pred[0]

        results.append({'sample_index': slide_name, 'label': idx_to_label[final_pred]})
        predicted_slides.add(slide_name)

# --- 6. SAFETY NET (Handling Missing Slides) ---
# Check for slides that were skipped in Block 1 because they had <10% tissue
all_test_files = [f for f in os.listdir(TEST_DIR) if f.startswith('img') and f.endswith(('.png', '.jpg'))]
missing_files = [f for f in all_test_files if f not in predicted_slides]

if len(missing_files) > 0:
    print(f"⚠️ Found {len(missing_files)} slides with no saved patches (Low Tissue).")
    print("🚑 Running Emergency Fallback (Center Crop) on missing slides...")

    with torch.no_grad():
        for filename in tqdm(missing_files):
            img_path = os.path.join(TEST_DIR, filename)

            # Find Mask Path
            mask_name = filename.replace('img', 'mask').replace('.jpg', '.png')
            mask_path = os.path.join(TEST_DIR, mask_name)
            if not os.path.exists(mask_path):
                 mask_path = os.path.join(TEST_DIR, filename.replace('img', 'mask'))

            # Use your ROBUST extraction function
            batch = extract_patches_with_fallback(img_path, mask_path, PATCH_SIZE, 56)

            if batch is None:
                # Corrupt file case
                results.append({'sample_index': filename, 'label': 'Luminal A'})
                continue

            # Predict
            batch = batch.to(DEVICE)
            with torch.amp.autocast(device_type=DEVICE.type, enabled=(DEVICE.type == 'cuda')):
                logits = model(batch)
                preds = torch.argmax(logits, dim=1).cpu().numpy()

            # Vote
            final_pred_idx = stats.mode(preds)[0]
            if isinstance(final_pred_idx, np.ndarray): final_pred_idx = final_pred_idx[0]

            results.append({'sample_index': filename, 'label': idx_to_label[final_pred_idx]})

# --- 7. SAVE ---
submission_df = pd.DataFrame(results)
submission_df = submission_df.sort_values(by='sample_index')
submission_df.to_csv("submission_robust.csv", index=False)

print(f"✅ Prediction complete! Generated {len(submission_df)} rows.")
print(submission_df.head())

🕵️ Predicting on 11607 saved patches...


100%|██████████| 91/91 [21:06<00:00, 13.92s/it]


🗳️ Voting...
⚠️ Found 114 slides with no saved patches (Low Tissue).
🚑 Running Emergency Fallback (Center Crop) on missing slides...


100%|██████████| 114/114 [01:11<00:00,  1.60it/s]

✅ Prediction complete! Generated 477 rows.
     sample_index      label
363  img_0000.png  Luminal A
364  img_0001.png    HER2(+)
0    img_0002.png  Luminal B
1    img_0003.png  Luminal B
2    img_0004.png    HER2(+)


In [ ]:
import torch.nn.functional as F

# --- HELPER: TTA PREDICTION FUNCTION ---
def predict_batch_with_tta(model, batch, device):
    """
    Predicts on a batch using 4-view TTA:
    1. Original
    2. Horizontal Flip
    3. Vertical Flip
    4. Rotate 90
    Returns averaged probabilities.
    """
    batch = batch.to(device)

    # 1. Original
    logits_orig = model(batch)
    probs_orig = F.softmax(logits_orig, dim=1)

    # 2. Horizontal Flip (Flip Width - Dim 3)
    batch_h = torch.flip(batch, dims=[3])
    logits_h = model(batch_h)
    probs_h = F.softmax(logits_h, dim=1)

    # 3. Vertical Flip (Flip Height - Dim 2)
    batch_v = torch.flip(batch, dims=[2])
    logits_v = model(batch_v)
    probs_v = F.softmax(logits_v, dim=1)

    # 4. Rotate 90 (Dims 2 and 3)
    batch_r = torch.rot90(batch, k=1, dims=[2, 3])
    logits_r = model(batch_r)
    probs_r = F.softmax(logits_r, dim=1)

    # Average the probabilities
    avg_probs = (probs_orig + probs_h + probs_v + probs_r) / 4.0

    # Return index of max probability
    return torch.argmax(avg_probs, dim=1).cpu().numpy()

# --- 5. INFERENCE LOOP (With TTA) ---

# A. Define Dataset for Saved Patches (Same as before)
class SavedPatchDataset(Dataset):
    def __init__(self, df, root_dir, transform=None):
        self.df = df
        self.root_dir = root_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.root_dir, "images", row['filename'])
        mask_path = os.path.join(self.root_dir, "masks", row['maskname'])

        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

        # Blackout
        _, binary_mask = cv2.threshold(mask, 127, 1, cv2.THRESH_BINARY)
        mask_3d = np.repeat(binary_mask[:, :, np.newaxis], 3, axis=2)
        image = image * mask_3d

        if self.transform:
            augmented = self.transform(image=image)
            image = augmented['image']

        return image, row['original_image']

# B. Create Loader
test_ds = SavedPatchDataset(test_df, TEMP_PATCH_DIR, transform=test_transform)
test_loader = DataLoader(test_ds, batch_size=128, shuffle=False, num_workers=2)

print(f"🕵️ Predicting on {len(test_df)} saved patches using TTA (4x views)...")

# C. Main Prediction Loop with TTA
patch_predictions = []
model.eval()

with torch.no_grad():
    for images, slide_names in tqdm(test_loader):
        # Use Mixed Precision + TTA Helper
        with torch.amp.autocast(device_type=DEVICE.type, enabled=(DEVICE.type == 'cuda')):
            preds = predict_batch_with_tta(model, images, DEVICE)

        # Store results
        for slide, pred in zip(slide_names, preds):
            patch_predictions.append({'sample_index': slide, 'pred': pred})

# D. Aggregation
print("🗳️ Voting...")
raw_df = pd.DataFrame(patch_predictions)
results = []
predicted_slides = set()

if not raw_df.empty:
    for slide_name, group in raw_df.groupby('sample_index'):
        final_pred = stats.mode(group['pred'])[0]
        if isinstance(final_pred, np.ndarray): final_pred = final_pred[0]

        results.append({'sample_index': slide_name, 'label': idx_to_label[final_pred]})
        predicted_slides.add(slide_name)

# --- 6. SAFETY NET (With TTA) ---
all_test_files = [f for f in os.listdir(TEST_DIR) if f.startswith('img') and f.endswith(('.png', '.jpg'))]
missing_files = [f for f in all_test_files if f not in predicted_slides]

if len(missing_files) > 0:
    print(f"⚠️ Found {len(missing_files)} slides with no saved patches.")
    print("🚑 Running Emergency Fallback (Center Crop + TTA)...")

    with torch.no_grad():
        for filename in tqdm(missing_files):
            img_path = os.path.join(TEST_DIR, filename)

            # Mask paths
            mask_name = filename.replace('img', 'mask').replace('.jpg', '.png')
            mask_path = os.path.join(TEST_DIR, mask_name)
            if not os.path.exists(mask_path):
                 mask_path = os.path.join(TEST_DIR, filename.replace('img', 'mask'))

            # Extract Batch
            batch = extract_patches_with_fallback(img_path, mask_path, PATCH_SIZE, 56)

            if batch is None:
                results.append({'sample_index': filename, 'label': 'Luminal A'})
                continue

            # Predict with TTA
            with torch.amp.autocast(device_type=DEVICE.type, enabled=(DEVICE.type == 'cuda')):
                preds = predict_batch_with_tta(model, batch, DEVICE)

            # Vote
            final_pred_idx = stats.mode(preds)[0]
            if isinstance(final_pred_idx, np.ndarray): final_pred_idx = final_pred_idx[0]

            results.append({'sample_index': filename, 'label': idx_to_label[final_pred_idx]})

# --- 7. SAVE ---
submission_df = pd.DataFrame(results)
submission_df = submission_df.sort_values(by='sample_index')
submission_df.to_csv("submission_tta.csv", index=False)

print(f"✅ TTA Prediction complete! Generated {len(submission_df)} rows.")
print(submission_df.head())

🕵️ Predicting on 11607 saved patches using TTA (4x views)...


  0%|          | 0/91 [00:00<?, ?it/s]